In [4]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from src.cleaning import OUTPUT

df = pd.read_csv(OUTPUT)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/lilyholmes/Documents/SUS/src/data/clean_runs.csv'

In [ ]:
df.columns

In [ ]:
df.groupby('gameName').count()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
# Check types and missing values before cleaning
df.info()
missing = df.isna().sum().sort_values(ascending=False)
pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(1)})

In [ ]:
missing[missing > 0].sort_values().plot.barh(figsize=(8, 5))
plt.xlabel("Missing rows")
plt.title("Missing values by column")
plt.tight_layout()
plt.show()

In [ ]:
# Check duplicate runs and values used for filtering
print("Duplicate run IDs:", df.runId.duplicated().sum())
print("Duplicate rows:", df.duplicated().sum())
display(df.loc[df.runId.duplicated(keep=False), ["runId", "gameId", "date", "primaryTime", "status"]].head(10))
print("Statuses:")
print(df.status.value_counts(dropna=False))
print("Emulated values:")
print(df.isEmulated.value_counts(dropna=False))

In [ ]:
# Zeros and very long times may need a closer look
time_cols = ["primaryTime", "realTime", "realTimeNoLoads", "inGameTime"]
print(df[time_cols].describe(percentiles=[.01, .5, .95, .99]).T)
print("Zero or negative times:")
print((df[time_cols] <= 0).sum())
print("Missing times:")
print(df[time_cols].isna().sum())

In [ ]:
# Limit the plot to the 99th percentile so long runs do not hide the shape
times = df.loc[df.primaryTime > 0, "primaryTime"]
times[times <= times.quantile(.99)].plot.hist(bins=50, figsize=(8, 4))
plt.xlabel("Primary time (seconds)")
plt.title("Run times through the 99th percentile")
plt.tight_layout()
plt.show()

In [ ]:
# Check dates that cannot be parsed and runs dated in the future
run_date = pd.to_datetime(df.date, errors="coerce")
verified_date = pd.to_datetime(df.verifiedDate, errors="coerce", utc=True)
print("Invalid run dates:", (df.date.notna() & run_date.isna()).sum())
print("Invalid verification dates:", (df.verifiedDate.notna() & verified_date.isna()).sum())
print("Future run dates:", (run_date > pd.Timestamp.today()).sum())
print("Date range:", run_date.min(), "to", run_date.max())